# 🎁 Bonus — Gradientenabstieg selbst gebaut

## Wie ein Modell wirklich lernt

**Das hier geht über den Workshop hinaus.** Es ist für alle gedacht, die mit dem Hauptnotebook
schneller durch sind oder wissen wollen, was `scikit-learn` im Verborgenen tut.

Im Hauptnotebook hat `LinearRegression` den tiefsten Punkt der Kostenschüssel für uns gefunden.
Hier findest du ihn **selbst** — mit demselben Verfahren, das auch neuronale Netze und LLMs
trainiert. Am Ende vergleichen wir: dieselben Zahlen, auf zwei Nachkommastellen.

**Voraussetzung:** das Hauptnotebook. Die Funktionen, die du dort selbst geschrieben hast, bekommst du hier
geschenkt, damit du dich auf das Neue konzentrieren kannst.

| Symbol | Bedeutung |
|:--:|---|
| 📖 | Erklärung — lesen |
| ▶️ | Fertiger Code — einfach ausführen |
| 🛠️ | **Challenge** — hier schreibst du selbst Code |
| ✅ | Selbsttest |
| 💡 | **Lösung** — zum Aufklappen, wenn du nicht weiterkommst |
| 💬 | Diskussionsfrage |

Es sind 4 Challenges.

---
## 0 · Setup

▶️ Alles aus dem Hauptnotebook auf einen Schlag: Werkzeuge, Daten, Aufteilung und die
Funktionen, die du dort selbst geschrieben hast.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Einheitliche Farben für alle Diagramme in diesem Notebook
BLAU, ORANGE, TEAL, GRAU = "#2563eb", "#e8590c", "#0d9488", "#6b7280"

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": GRAU,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#e5e7eb",
    "grid.linewidth": 0.8,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "font.size": 11,
})
np.set_printoptions(suppress=True, precision=2)

print("Setup fertig ✔")

In [ ]:
def lade_daten(dateiname):
    """Liest eine der Workshop-CSVs ein — lokal oder in Google Colab."""
    kandidaten = [
        Path(dateiname),
        Path("data") / dateiname,
        Path("..") / "data" / dateiname,
        Path("challenges/lineare-regression/data") / dateiname,
    ]
    for pfad in kandidaten:
        if pfad.exists():
            print(f"Gelesen: {pfad}")
            return pd.read_csv(pfad)

    # Google Colab: Datei von Hand hochladen
    try:
        from google.colab import files
        print(f"'{dateiname}' nicht gefunden — bitte jetzt hochladen:")
        files.upload()
        return pd.read_csv(dateiname)
    except ImportError:
        raise FileNotFoundError(
            f"'{dateiname}' nicht gefunden. Lege die CSV neben dieses Notebook."
        )


df = lade_daten("haus_preise_einfach.csv")

# Dieselbe Aufteilung wie im Hauptnotebook
zufall = np.random.default_rng(42)
gemischt = zufall.permutation(len(df))
grenze = int(0.8 * len(df))

train, test = df.iloc[gemischt[:grenze]], df.iloc[gemischt[grenze:]]
x_train = train["wohnflaeche_qm"].to_numpy()
y_train = train["preis_usd"].to_numpy().astype(float)
x_test = test["wohnflaeche_qm"].to_numpy()
y_test = test["preis_usd"].to_numpy().astype(float)


def vorhersage(x, w, b):
    """Aus dem Hauptnotebook: preis = w * wohnflaeche + b"""
    return w * x + b


def mse(y_wahr, y_vorhersage):
    """Aus dem Hauptnotebook: mittlerer quadratischer Fehler"""
    return float(np.mean((y_vorhersage - y_wahr) ** 2))


print(f"{len(x_train)} Trainingshäuser, {len(x_test)} Testhäuser — bereit.")

---
## 1 · Warum überhaupt ein Suchverfahren?

📖 Im Hauptnotebook haben wir 200 Werte für $w$ durchprobiert und den besten genommen. Das
funktioniert — solange es nur *einen* Parameter gibt.

Rechnen wir kurz nach, was "alles durchprobieren" wirklich kostet:

| Modell | Parameter | Kombinationen bei 100 Werten je Parameter |
|---|---|---|
| unsere Gerade | 2 | 100² = 10.000 ✔ machbar |
| kleines neuronales Netz | 25.000 | 100²⁵⁰⁰⁰ ✘ |
| GPT-3 | 175.000.000.000 | ✘✘✘ |

Ausprobieren skaliert nicht. Wir brauchen ein Verfahren, das den Weg nach unten **findet**,
statt ihn zu suchen — und das an jeder Stelle nur wissen muss, in welche Richtung es bergab geht.

---
## 2 · Der Gradient: In welche Richtung geht es bergab?

📖 Stell dir vor, du stehst im Nebel auf einem Hang und willst ins Tal. Du siehst das Tal nicht,
aber du spürst unter den Füßen, in welche Richtung es **abwärts** geht — und machst einen Schritt
dorthin. Dann wieder. Und wieder.

Der **Gradient** ist genau dieses Gefühl unter den Füßen: die Steigung der Kostenfunktion. Er
zeigt bergauf, also gehen wir in die **Gegenrichtung**.

Für unser Modell lassen sich die beiden Steigungen exakt ausrechnen (Ableitung des MSE nach $w$
bzw. nach $b$):

$$\frac{\partial \text{MSE}}{\partial w} = \frac{2}{n}\sum_{i=1}^{n}(\hat{y}_i - y_i)\cdot x_i
\qquad
\frac{\partial \text{MSE}}{\partial b} = \frac{2}{n}\sum_{i=1}^{n}(\hat{y}_i - y_i)$$

In Worten:

* **`dw`**: Fehler mal Eingabewert, gemittelt. Ein Haus mit großer Fläche zieht $w$ stärker.
* **`db`**: nur der mittlere Fehler. Sagen wir im Schnitt zu wenig voraus, wandert $b$ nach oben.

### 🛠️ Bonus-Challenge 1 — Die Gradienten

Setze die beiden Formeln in Code um. Die Funktion gibt **zwei** Werte zurück: `dw` und `db`.

*Tipp: `fehler = vorhersage(x, w, b) - y` ist ein Array. `np.sum(fehler * x)` und
`np.sum(fehler)` liefern die Summen. `n = len(x)`.*

In [ ]:
def gradienten(x, y, w, b):
    """Steigung der Kostenfunktion an der Stelle (w, b).

    Rückgabe: (dw, db)
    """
    n = len(x)
    # TODO
    raise NotImplementedError("Bonus-Challenge 1: gradienten() implementieren")

In [ ]:
# ✅ Selbsttest — wir prüfen deine Formel gegen die numerische Steigung.
# (Kosten leicht links und rechts von w messen und die Differenz durch den Abstand teilen.)
x_probe = np.array([1.0, 2.0, 3.0])
y_probe = np.array([2.0, 4.0, 7.0])
dw, db = gradienten(x_probe, y_probe, 1.0, 0.0)

h = 1e-6
dw_numerisch = (mse(y_probe, vorhersage(x_probe, 1.0 + h, 0.0))
                - mse(y_probe, vorhersage(x_probe, 1.0 - h, 0.0))) / (2 * h)
db_numerisch = (mse(y_probe, vorhersage(x_probe, 1.0, 0.0 + h))
                - mse(y_probe, vorhersage(x_probe, 1.0, 0.0 - h))) / (2 * h)

assert np.isclose(dw, dw_numerisch, atol=1e-3), f"dw stimmt nicht: {dw} statt {dw_numerisch}"
assert np.isclose(db, db_numerisch, atol=1e-3), f"db stimmt nicht: {db} statt {db_numerisch}"
print(f"dw = {dw:.3f} (numerisch {dw_numerisch:.3f})")
print(f"db = {db:.3f} (numerisch {db_numerisch:.3f})")
print("✅ Bonus-Challenge 1 gelöst")

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def gradienten(x, y, w, b):
    """Steigung der Kostenfunktion an der Stelle (w, b).

    Rückgabe: (dw, db)
    """
    n = len(x)
    fehler = vorhersage(x, w, b) - y
    dw = (2 / n) * np.sum(fehler * x)
    db = (2 / n) * np.sum(fehler)
    return dw, db
```

Die beiden Formeln stehen 1:1 im Code. `fehler * x` multipliziert elementweise — jedes Haus
steuert seinen Fehler mal seiner Fläche bei, und `np.sum` addiert alles zusammen.

</details>

---
## 3 · Erster Trainingsversuch — und warum er scheitert

📖 Wir haben alles zusammen. Ein Schritt bergab sieht so aus:

```
w = w - lernrate * dw
b = b - lernrate * db
```

Die **Lernrate** ist die Schrittweite. Nehmen wir 0.1 und machen acht Schritte.

▶️ Ausführen und die Zahlen anschauen.

In [ ]:
w, b = 0.0, 0.0
lernrate = 0.1

print(f"{'Schritt':>7} {'w':>13} {'b':>13} {'MSE':>13}")
for schritt in range(1, 9):
    kosten = mse(y_train, vorhersage(x_train, w, b))
    dw, db = gradienten(x_train, y_train, w, b)
    w = w - lernrate * dw
    b = b - lernrate * db
    print(f"{schritt:>7} {w:>13.2e} {b:>13.2e} {kosten:>13.2e}")

💬 **Was ist da passiert?**

<details>
<summary>Antwort aufklappen</summary>

Das Modell ist **explodiert**. Statt ins Tal zu laufen, springt es mit jedem Schritt weiter aus
der Schüssel heraus — die Zahlen wechseln das Vorzeichen und werden immer größer.

Der Grund steckt in den Größenordnungen. Der Gradient `dw` ist ungefähr
`2 · mittlerer Fehler · mittlere Fläche` ≈ `2 · 500.000 · 200` = **200 Millionen**. Mal Lernrate
0.1 macht das einen Schritt von 20 Millionen — für einen Wert, dessen Optimum bei etwa 3.000
liegt. Wir schießen meilenweit über das Ziel hinaus, landen auf der anderen Seite noch höher
am Hang, und der nächste Schritt wird noch größer.

Zwei Auswege:
1. Eine **winzige Lernrate** (etwa 0.00001). Funktioniert, ist aber Herumprobieren — und beim
   nächsten Datensatz beginnt die Sucherei von vorn.
2. Die **Daten in eine handliche Größenordnung bringen**. Das ist der Standardweg.
</details>

### Feature Scaling: alle Merkmale auf dieselbe Skala

📖 Wir rechnen jede Wohnfläche in eine **Standardabweichung um den Mittelwert** um
(*Standardisierung*, *z-Wert*):

$$x_{\text{skaliert}} = \frac{x - \mu}{\sigma}$$

Danach ist der Mittelwert 0 und die Standardabweichung 1. Aus "193 m²" wird "0", aus "279 m²"
wird "1" (eine Standardabweichung über dem Schnitt). Die Information bleibt dieselbe, nur die
Zahlen sind jetzt handlich — und dieselbe Lernrate funktioniert für jedes Merkmal.

⚠️ **Die wichtigste Regel dabei:** $\mu$ und $\sigma$ werden **nur aus den Trainingsdaten**
berechnet. Die Testdaten werden mit *denselben* Werten umgerechnet. Sonst fließt Wissen aus den
Testdaten ins Training — der Klassiker unter den Anfängerfehlern (*data leakage*).

### 🛠️ Bonus-Challenge 2 — Standardisieren

Zwei Teile: die Funktion schreiben — und sie richtig anwenden.

In [ ]:
def standardisiere(werte, mittelwert, standardabweichung):
    """Rechnet Werte in z-Werte um: (Wert - Mittelwert) / Standardabweichung."""
    # TODO
    raise NotImplementedError("Bonus-Challenge 2: standardisiere() implementieren")


# TODO: Mittelwert und Standardabweichung der TRAININGSDATEN berechnen
# Tipp: x_train.mean() und x_train.std()
mittelwert_x = ...
std_x = ...

# TODO: Trainings- UND Testdaten umrechnen — beide mit den Werten von oben!
x_train_s = ...
x_test_s = ...

print(f"Trainingsdaten: μ = {mittelwert_x:.2f} m², σ = {std_x:.2f} m²")
print(f"Nach dem Skalieren: Mittelwert = {x_train_s.mean():.6f}, Std = {x_train_s.std():.6f}")

In [ ]:
# ✅ Selbsttest
assert np.isclose(x_train_s.mean(), 0, atol=1e-9), "Trainingsmittelwert muss 0 sein"
assert np.isclose(x_train_s.std(), 1, atol=1e-9), "Trainings-Std muss 1 sein"
assert np.allclose(x_test_s, (x_test - mittelwert_x) / std_x), \
    "Die Testdaten müssen mit den Werten der TRAININGSdaten skaliert werden"
print(f"Ein 300-m²-Haus entspricht z = {standardisiere(300, mittelwert_x, std_x):.2f}")
print("✅ Bonus-Challenge 2 gelöst")

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def standardisiere(werte, mittelwert, standardabweichung):
    """Rechnet Werte in z-Werte um: (Wert - Mittelwert) / Standardabweichung."""
    return (werte - mittelwert) / standardabweichung


mittelwert_x = x_train.mean()
std_x = x_train.std()

x_train_s = standardisiere(x_train, mittelwert_x, std_x)
x_test_s = standardisiere(x_test, mittelwert_x, std_x)

print(f"Trainingsdaten: μ = {mittelwert_x:.2f} m², σ = {std_x:.2f} m²")
print(f"Nach dem Skalieren: Mittelwert = {x_train_s.mean():.6f}, Std = {x_train_s.std():.6f}")
```

Der entscheidende Teil ist nicht die Formel, sondern die vorletzte Zeile: `x_test_s` wird mit
`mittelwert_x` und `std_x` aus den **Trainings**daten gerechnet, nicht mit eigenen. Alles
andere wäre data leakage.

</details>

---
## 4 · Die Trainingsschleife

📖 Das ist das Herzstück. Jedes neuronale Netz, jedes LLM wird im Prinzip mit genau dieser
Schleife trainiert:

```
starte mit w = 0 und b = 0
wiederhole für jede Epoche:
    1. Kosten berechnen und protokollieren   (wo stehe ich?)
    2. Gradienten berechnen                  (wo geht es bergab?)
    3. w und b einen kleinen Schritt weiter  (Schritt machen)
```

Eine **Epoche** ist ein Durchlauf über alle Trainingsdaten.

### 🛠️ Bonus-Challenge 3 — Die Trainingsschleife

Baue die Schleife. Sie gibt drei Dinge zurück: das gelernte `w`, das gelernte `b` und den
`verlauf` — eine Liste mit dem MSE nach jeder Epoche, damit wir dem Modell beim Lernen zusehen
können.

Alle Bausteine hast du: `vorhersage`, `mse`, `gradienten`.

In [ ]:
def trainiere(x, y, lernrate=0.1, epochen=200):
    """Sucht mit Gradientenabstieg gute Werte für w und b.

    Rückgabe: (w, b, verlauf)
    """
    w = 0.0
    b = 0.0
    verlauf = []

    for epoche in range(epochen):
        # TODO 1: aktuelle Kosten berechnen und an verlauf anhängen
        # TODO 2: Gradienten berechnen
        # TODO 3: w und b aktualisieren (Schritt in die Gegenrichtung des Gradienten)
        raise NotImplementedError("Bonus-Challenge 3: trainiere() implementieren")

    return w, b, verlauf


w_gelernt, b_gelernt, verlauf = trainiere(x_train_s, y_train, lernrate=0.1, epochen=200)

print(f"w = {w_gelernt:,.2f}")
print(f"b = {b_gelernt:,.2f}")
print(f"MSE am Anfang: {verlauf[0]:,.0f}")
print(f"MSE am Ende:   {verlauf[-1]:,.0f}")

In [ ]:
# ✅ Selbsttest — Vergleich mit der exakt berechneten Optimallösung
matrix = np.c_[x_train_s, np.ones(len(x_train_s))]
w_optimal, b_optimal = np.linalg.lstsq(matrix, y_train, rcond=None)[0]

assert len(verlauf) == 200, "verlauf sollte einen MSE-Wert pro Epoche enthalten"
assert verlauf[0] > verlauf[-1], "Die Kosten sollten sinken"
assert np.isclose(w_gelernt, w_optimal, rtol=1e-3), f"w={w_gelernt:.1f}, optimal wäre {w_optimal:.1f}"
assert np.isclose(b_gelernt, b_optimal, rtol=1e-3), f"b={b_gelernt:.1f}, optimal wäre {b_optimal:.1f}"
print(f"Optimum (exakt berechnet): w = {w_optimal:,.2f}, b = {b_optimal:,.2f}")
print("✅ Bonus-Challenge 3 gelöst — dein Gradientenabstieg findet das Optimum")

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def trainiere(x, y, lernrate=0.1, epochen=200):
    """Sucht mit Gradientenabstieg gute Werte für w und b.

    Rückgabe: (w, b, verlauf)
    """
    w = 0.0
    b = 0.0
    verlauf = []

    for epoche in range(epochen):
        verlauf.append(mse(y, vorhersage(x, w, b)))
        dw, db = gradienten(x, y, w, b)
        w = w - lernrate * dw
        b = b - lernrate * db

    return w, b, verlauf


w_gelernt, b_gelernt, verlauf = trainiere(x_train_s, y_train, lernrate=0.1, epochen=200)

print(f"w = {w_gelernt:,.2f}")
print(f"b = {b_gelernt:,.2f}")
print(f"MSE am Anfang: {verlauf[0]:,.0f}")
print(f"MSE am Ende:   {verlauf[-1]:,.0f}")
```

Drei Zeilen im Schleifenkörper — mehr ist Training nicht. Wichtig ist das Minus in
`w - lernrate * dw`: Der Gradient zeigt *bergauf*, wir wollen bergab.

</details>

▶️ **Die Lernkurve** — so sieht Lernen aus:

In [ ]:
fig, ax = plt.subplots()
ax.plot(verlauf, color=BLAU, linewidth=2)
ax.set_xlabel("Epoche")
ax.set_ylabel("MSE (Trainingsdaten)")
ax.set_title("Die Kosten fallen — das Modell lernt")
ax.text(60, verlauf[0] * 0.75, "steiler Abfall:\ngroße Gradienten", color=GRAU)
ax.text(120, verlauf[-1] * 1.25, "flach: fast im Minimum angekommen", color=GRAU)
plt.show()

### Die Lernrate ist die wichtigste Stellschraube

▶️ Dreimal dasselbe Training, nur mit unterschiedlicher Schrittweite:

In [ ]:
fig, ax = plt.subplots()
for lernrate, farbe, beschriftung in [
    (0.001, GRAU, "0.001 — zu klein: kommt kaum vom Fleck"),
    (0.1, BLAU, "0.1 — passt"),
    (1.1, ORANGE, "1.1 — zu groß: explodiert"),
]:
    _, _, v = trainiere(x_train_s, y_train, lernrate=lernrate, epochen=60)
    ax.plot(v, color=farbe, linewidth=2, label=beschriftung)

ax.set_yscale("log")
ax.set_xlabel("Epoche")
ax.set_ylabel("MSE (logarithmisch)")
ax.set_title("Drei Lernraten, dasselbe Modell")
ax.legend()
plt.show()

---
## 5 · Die Probe aufs Exempel

📖 Unser gelerntes $w$ gehört zu den *skalierten* Flächen. Rechnen wir zurück in USD pro m²:

$$\text{preis} = w_s \cdot \frac{x - \mu}{\sigma} + b_s
= \underbrace{\frac{w_s}{\sigma}}_{w_{\text{echt}}} \cdot x
+ \underbrace{\left(b_s - \frac{w_s \cdot \mu}{\sigma}\right)}_{b_{\text{echt}}}$$

▶️ Und dann der Moment der Wahrheit — Vergleich mit `scikit-learn` aus dem Hauptnotebook:

In [ ]:
from sklearn.linear_model import LinearRegression

w_echt = w_gelernt / std_x
b_echt = b_gelernt - w_gelernt * mittelwert_x / std_x

modell = LinearRegression().fit(x_train.reshape(-1, 1), y_train)

print(f"{'':<28}{'w (USD/m²)':>14}{'b (USD)':>16}")
print("-" * 58)
print(f"{'Dein Gradientenabstieg':<28}{w_echt:>14,.2f}{b_echt:>16,.2f}")
print(f"{'scikit-learn':<28}{modell.coef_[0]:>14,.2f}{modell.intercept_:>16,.2f}")
print()
print("Dieselben Zahlen. Keine Magie in der Bibliothek — nur diese Schleife. ✔")

---
## 6 · Mehrere Merkmale: dieselbe Schleife, vektorisiert

📖 Ein Modell kann mehrere Merkmale gleichzeitig benutzen — jedes bekommt sein eigenes Gewicht,
und alle Beiträge werden addiert. (Ausführlich erklärt ist das im Notebook
`bonus_mehrere_merkmale_challenge.ipynb`; für hier reicht die Kurzfassung.) Auch dafür brauchst
du keine Bibliothek — der Code ändert sich kaum.

Statt einzelner Zahlen rechnen wir mit einer **Matrix** `X` (eine Zeile pro Haus, eine Spalte pro
Merkmal) und einem **Vektor** `w`. Das Skalarprodukt `X @ w` erledigt alle Multiplikationen auf
einmal:

$$\text{preis} = w_1 x_1 + w_2 x_2 + w_3 x_3 + b = \mathbf{w} \cdot \mathbf{x} + b$$

In [ ]:
df_multi = lade_daten("haus_preise_multi.csv")
MERKMALE = ["wohnflaeche_qm", "qualitaet", "baujahr"]

train_multi = df_multi.iloc[gemischt[:grenze]]
test_multi = df_multi.iloc[gemischt[grenze:]]
X_train = train_multi[MERKMALE].to_numpy(dtype=float)
X_test = test_multi[MERKMALE].to_numpy(dtype=float)

# Standardisieren — jede Spalte mit ihrem eigenen Mittelwert und ihrer eigenen Std,
# beide wieder nur aus den Trainingsdaten. axis=0 heißt "spaltenweise".
mittelwerte = X_train.mean(axis=0)
stds = X_train.std(axis=0)
X_train_s = standardisiere(X_train, mittelwerte, stds)
X_test_s = standardisiere(X_test, mittelwerte, stds)

print(f"X_train hat die Form {X_train.shape}  (Häuser × Merkmale)")
print(f"Mittelwerte:  {mittelwerte}")
print(f"Standardabw.: {stds}")
print()
print("Ohne Skalierung reichen die Zahlen von 1 (Qualität) bis 2015 (Baujahr).")
print("Eine einzige Lernrate könnte für beide niemals gleichzeitig passen.")

### 🛠️ Bonus-Challenge 4 — Training mit mehreren Merkmalen

Dieselbe Schleife wie eben — nur ist `w` jetzt ein **Vektor** statt einer Zahl.

| einzeln | mehrere Merkmale |
|---|---|
| `w = 0.0` | `w = np.zeros(anzahl_merkmale)` |
| `w * x + b` | `X @ w + b` |
| `np.sum(fehler * x)` | `X.T @ fehler` |

`X.T` ist die transponierte Matrix — dadurch liefert das Skalarprodukt für **jedes** Merkmal
seine eigene Steigung, alle auf einmal.

In [ ]:
def trainiere_multi(X, y, lernrate=0.1, epochen=300):
    """Gradientenabstieg für beliebig viele Merkmale.

    Rückgabe: (w, b, verlauf) — w ist ein Vektor mit einem Gewicht pro Merkmal.
    """
    n, anzahl_merkmale = X.shape
    w = np.zeros(anzahl_merkmale)
    b = 0.0
    verlauf = []

    for epoche in range(epochen):
        # TODO 1: fehler = Vorhersage minus Wahrheit   (Tipp: X @ w + b - y)
        # TODO 2: Kosten protokollieren                (Tipp: np.mean(fehler ** 2))
        # TODO 3: dw und db berechnen
        # TODO 4: w und b aktualisieren
        raise NotImplementedError("Bonus-Challenge 4: trainiere_multi() implementieren")

    return w, b, verlauf


w_multi, b_multi, verlauf_multi = trainiere_multi(X_train_s, y_train, lernrate=0.1, epochen=300)

for merkmal, gewicht in zip(MERKMALE, w_multi):
    print(f"  {merkmal:<16} w = {gewicht:>12,.0f}")
print(f"  {'(bias)':<16} b = {b_multi:>12,.0f}")

In [ ]:
# ✅ Selbsttest — wieder gegen die exakt berechnete Optimallösung
matrix = np.c_[X_train_s, np.ones(len(X_train_s))]
optimal = np.linalg.lstsq(matrix, y_train, rcond=None)[0]

assert len(verlauf_multi) == 300, "Ein MSE-Wert pro Epoche"
assert verlauf_multi[0] > verlauf_multi[-1], "Die Kosten sollten sinken"
assert np.allclose(w_multi, optimal[:-1], rtol=1e-3), f"w stimmt nicht: {w_multi} statt {optimal[:-1]}"
assert np.isclose(b_multi, optimal[-1], rtol=1e-3), f"b stimmt nicht: {b_multi} statt {optimal[-1]}"
print("✅ Bonus-Challenge 4 gelöst — dieselbe Schleife, jetzt in drei Dimensionen")

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def trainiere_multi(X, y, lernrate=0.1, epochen=300):
    """Gradientenabstieg für beliebig viele Merkmale.

    Rückgabe: (w, b, verlauf) — w ist ein Vektor mit einem Gewicht pro Merkmal.
    """
    n, anzahl_merkmale = X.shape
    w = np.zeros(anzahl_merkmale)
    b = 0.0
    verlauf = []

    for epoche in range(epochen):
        fehler = X @ w + b - y
        verlauf.append(float(np.mean(fehler ** 2)))
        dw = (2 / n) * (X.T @ fehler)
        db = (2 / n) * np.sum(fehler)
        w = w - lernrate * dw
        b = b - lernrate * db

    return w, b, verlauf


w_multi, b_multi, verlauf_multi = trainiere_multi(X_train_s, y_train, lernrate=0.1, epochen=300)

for merkmal, gewicht in zip(MERKMALE, w_multi):
    print(f"  {merkmal:<16} w = {gewicht:>12,.0f}")
print(f"  {'(bias)':<16} b = {b_multi:>12,.0f}")
```

Vergleiche mit `trainiere()`: Geändert hat sich nur `X @ w` statt `w * x` und `X.T @ fehler`
statt `np.sum(fehler * x)`. Die Schleife selbst ist identisch — dieselben vier Schritte.

</details>

In [ ]:
# ▶️ Zurückgerechnet in echte Einheiten — und noch einmal die Gegenprobe
w_multi_echt = w_multi / stds
b_multi_echt = b_multi - np.sum(w_multi * mittelwerte / stds)

modell_multi = LinearRegression().fit(X_train, y_train)

print(f"{'Merkmal':<18}{'Du':>16}{'scikit-learn':>18}")
print("-" * 52)
for merkmal, deins, sklearn_wert in zip(MERKMALE, w_multi_echt, modell_multi.coef_):
    print(f"{merkmal:<18}{deins:>16,.2f}{sklearn_wert:>18,.2f}")
print(f"{'(bias)':<18}{b_multi_echt:>16,.2f}{modell_multi.intercept_:>18,.2f}")

---
## Was du gebaut hast

Ohne eine einzige Zeile Machine-Learning-Bibliothek:

1. **Gradient** — `gradienten()`: die Richtung, in der es bergab geht
2. **Feature Scaling** — `standardisiere()`: Merkmale in eine handliche Größenordnung bringen
3. **Trainingsschleife** — `trainiere()`: viele kleine Schritte bergab
4. **Vektorisierung** — `trainiere_multi()`: dieselbe Schleife für beliebig viele Merkmale

Das ist der Kern jedes Trainings im Machine Learning. Was bei einem LLM anders ist:

* Millionen bis Milliarden Parameter statt zwei — aber dieselbe Schleife.
* Die Gradienten werden nicht von Hand hergeleitet, sondern automatisch berechnet
  (*Backpropagation*, PyTorch macht das für dich).
* Pro Schritt wird nicht der ganze Datensatz benutzt, sondern eine zufällige Teilmenge
  (*Mini-Batch*) — sonst dauert eine einzige Epoche Tage.
* Statt einer festen Lernrate laufen ausgefeiltere Varianten (*Adam*), die die Schrittweite
  selbst anpassen.

Alles Verfeinerungen desselben Prinzips: schau, wo es bergab geht, und mach einen Schritt.

---
## Noch mehr?

1. **Ab welcher Lernrate explodiert `trainiere()`?** Probiere 0.9, 0.99, 1.0, 1.01. Warum liegt
   die Grenze genau dort? (Tipp: bei standardisierten Daten hat sie mit der Zahl 1 zu tun.)
2. **Startwerte.** Was passiert, wenn `trainiere()` nicht bei `w = 0`, sondern bei
   `w = 500_000` startet? Findet der Gradientenabstieg trotzdem dasselbe Optimum?
3. **Mini-Batch.** Baue `trainiere()` so um, dass pro Schritt nur 64 zufällige Häuser benutzt
   werden. Wie sieht die Lernkurve jetzt aus — und warum ist sie so zackig?
4. **Ohne Skalierung.** Finde von Hand eine Lernrate, mit der das Training auf den *unskalierten*
   Daten funktioniert. Wie viele Epochen brauchst du dann, bis `b` in der Nähe von −39.437 ist?